# Paper 1: Context-Aware Video Summarization via Scored Multimodal Transcripts

This notebook implements the advanced video summarization pipeline incorporating:
1.  **Multimodal Analysis**: Processing both audio and visual streams.
2.  **Content-Aware Frame Classification**: Identifying frames with text, diagrams, formulas, and code.
3.  **Frame Importance Scoring**: Quantifying the significance of each visual frame.
4.  **Specialized Content Extraction**: Using the best tool for each content type (e.g., LaTeX-OCR for formulas).
5.  **Multimodal-Augmented Transcript (MAT)**: A rich, structured representation of the video.
6.  **Advanced Summarization**: Using the MAT to generate a high-quality summary with a Large Language Model.

## Phase 0: Setup and Installation

In [ ]:
# Install all required libraries and dependencies
!pip install -q opencv-python  pydub
!pip install -q openai-whisper
!pip install -q transformers torch pillow
!pip install -q tensorflow scikit-learn
# Specialized OCR libraries
!sudo apt-get install tesseract-ocr
!pip install -q pytesseract
!pip install -q "pix2tex[api]"
!pip install -q easyocr
# LLM and YouTube downloader
!pip install -q yt-dlp
!pip install -q moviepy
!pip install -q google-generativeai
!pip install paddleocr
# Add this line with the other !pip install commands
!pip install -q rouge-score


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.7/824.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.0/427.0 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 89.9 MB/s eta 0:00:00
   ━

In [ ]:
# Import all necessary libraries
import os
import cv2
import json
import numpy as np
import re
import yt_dlp
import whisper
import tensorflow as tf
from moviepy.editor import VideoFileClip
import google.generativeai as genai
from PIL import Image
import pytesseract
from pix2tex.cli import LatexOCR
from transformers import pipeline
from google.colab import drive
import easyocr

# Mount Google Drive to save models and data persistently
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create project directory structure in Google Drive
PROJECT_DIR = '/content/drive/MyDrive/AI_Paper1_Project/'
VIDEO_DIR = os.path.join(PROJECT_DIR, 'videos')
CLASSIFIER_DATA_DIR = os.path.join(PROJECT_DIR, 'classifier_data')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs')

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(CLASSIFIER_DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

genai.configure(api_key="GOOGLE_API_KEY")

print(f"Project directory created at: {PROJECT_DIR}")

Project directory created at: /content/drive/MyDrive/AI_Paper1_Project/


## Phase 1: Frame Classifier Training

This is a one-time setup phase. We will extract frames from sample videos, manually label them, and then train a simple CNN to classify them automatically.

**YOUR ACTION REQUIRED:**
1.  Download 2-3 sample videos covering different topics (code, math, etc.) and place them in the `videos` folder in your Google Drive.
2.  Run the next cell to extract frames from these videos.
3.  **Manually** go into the `classifier_data` folder and create subfolders: `text`, `diagram`, `formula`, `code`.
4.  **Manually** sort the extracted frames into these four folders. Aim for at least 50-100 images per category. This creates your labeled dataset.
5.  Once you have sorted the images, run the cell to train the classifier.

In [ ]:
# --- Helper cell to extract frames for labeling ---
# --- Run this once to generate images for manual sorting ---

def extract_frames_for_labeling(video_dir, output_dir, frames_per_vid=200):
    os.makedirs(output_dir, exist_ok=True)
    for video_file in os.listdir(video_dir):
        if not video_file.endswith(('.mp4', '.mkv')):
            continue
        video_path = os.path.join(video_dir, video_file)
        print(f"Extracting frames from {video_file}...")

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        interval = total_frames // frames_per_vid if frames_per_vid > 0 else total_frames

        count = 0
        while cap.isOpened() and count < total_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, count)
            success, image = cap.read()
            if not success:
                break

            frame_name = f"{os.path.splitext(video_file)[0]}_frame_{count}.jpg"
            cv2.imwrite(os.path.join(output_dir, frame_name), image)
            count += interval

        cap.release()
    print("Frame extraction for labeling complete.")
    print(f"Please go to {output_dir}, create subfolders (text, diagram, formula, code) and sort the images.")

# Run the function
#extract_frames_for_labeling(VIDEO_DIR, CLASSIFIER_DATA_DIR)

In [ ]:
# --- Cell to Train the Frame Classifier ---
# --- Run this AFTER you have manually sorted your images ---

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
CLASSIFIER_PATH = os.path.join(MODELS_DIR, 'frame_classifier.h5')

def train_classifier():
    # Check if data exists
    if not os.path.exists(os.path.join(CLASSIFIER_DATA_DIR, 'text')):
        print("Classifier training skipped. Labeled data not found.")
        print(f"Please create subfolders in {CLASSIFIER_DATA_DIR} and add labeled images.")
        return None

    print("Loading labeled image data...")
    # Load datasets
    train_ds = tf.keras.utils.image_dataset_from_directory(
        CLASSIFIER_DATA_DIR,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        CLASSIFIER_DATA_DIR,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )

    class_names = train_ds.class_names
    print(f"Found classes: {class_names}")

    # Configure dataset for performance
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

    # Build the model
    base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
    base_model.trainable = False

    model = tf.keras.Sequential([
        tf.keras.layers.Rescaling(1./255, input_shape=IMG_SIZE + (3,)),
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(len(class_names), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    print("Training classifier...")
    history = model.fit(train_ds, validation_data=val_ds, epochs=10)

    # Save the trained model and class names
    model.save(CLASSIFIER_PATH)
    class_names_path = os.path.join(MODELS_DIR, 'class_names.json')
    with open(class_names_path, 'w') as f:
        json.dump(class_names, f)

    print(f"Classifier trained and saved to {CLASSIFIER_PATH}")
    return model, class_names

# Uncomment the line below to run the training
#train_classifier()

## Phase 2: Core Pipeline Components

In [ ]:
# --- Component 0: Helper Functions ---

def download_video(url, output_dir):
    """Downloads a YouTube video to the specified directory."""
    ydl_opts = {
        'format': 'best[ext=mp4]/best',
        'outtmpl': os.path.join(output_dir, '%(title)s.%(ext)s'),
        'quiet': True,
        'restrictfilenames': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)

def decompose_video(video_path, output_dir):
    """Decomposes video into audio and a sequence of frame paths with timestamps."""
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    audio_path = os.path.join(output_dir, f"{base_name}.wav")
    frames_dir = os.path.join(output_dir, f"{base_name}_frames")
    os.makedirs(frames_dir, exist_ok=True)

    # 1. Extract Audio using pydub
    print("Extracting audio...")
    from pydub import AudioSegment
    audio = AudioSegment.from_file(video_path, "mp4")
    audio.export(audio_path, format="wav")


    # 2. Extract Frames
    print("Extracting frames...")
    frame_paths = []
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * 20)
    frame_count = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        if frame_count % frame_interval == 0:
            timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
            frame_path = os.path.join(frames_dir, f"frame_{timestamp_ms}.jpg")
            cv2.imwrite(frame_path, frame)
            frame_paths.append({'path': frame_path, 'timestamp': timestamp_ms})

        frame_count += 1

    cap.release()
    return audio_path, frame_paths

In [ ]:
# --- Component 1: Audio Processor (ASR) ---
import torch # Import torch to check for CUDA availability
import whisper # Import whisper library

def process_audio_stream(audio_path):
    """Transcribes audio using Whisper and returns segments with timestamps."""
    print("Loading Whisper model for transcription...")
    # Use a smaller model for faster processing in Colab
    # Check for CUDA availability and set device accordingly
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    # Explicitly map location to the determined device
    model = whisper.load_model("base.en", device=device)
    print("Transcribing audio (this may take a while)...")
    result = model.transcribe(audio_path, word_timestamps=True)
    return result['segments']

In [ ]:
# --- Component 2: Visual Processor (Core Novelty) ---

# Import necessary libraries
import cv2
import numpy as np
import easyocr
from PIL import Image
import tensorflow as tf
import os
import json
import re
from pix2tex.cli import LatexOCR
from transformers import pipeline
from concurrent.futures import ProcessPoolExecutor, as_completed

# Initialize EasyOCR
try:
    print("Loading EasyOCR model...")
    reader = easyocr.Reader(['en'])
    print("EasyOCR model loaded successfully.")
except Exception as e_easyocr:
    print(f"Error loading EasyOCR model: {e_easyocr}")
    reader = None
    print("WARNING: EasyOCR could not be loaded. Text recognition will not be available.")


# try:
#     frame_classifier = tf.keras.models.load_model(CLASSIFIER_PATH)
#     with open(os.path.join(MODELS_DIR, 'class_names.json'), 'r') as f:
#         CLASS_NAMES = json.load(f)
# except IOError:
frame_classifier = None
CLASS_NAMES = ['text', 'diagram', 'formula', 'code'] # Default
print("WARNING: Trained frame classifier not found or could not be loaded. Using default classification.")

latex_ocr_model = LatexOCR()
captioning_pipeline = pipeline("image-to-text", model="salesforce/blip-image-captioning-base")


print("Loading other visual models...")
# Check if the cascade classifier file exists before loading
face_cascade_path = os.path.join(cv2.data.haarcascades, 'haarcascade_frontalface_default.xml')
if os.path.exists(face_cascade_path):
    face_cascade = cv2.CascadeClassifier(face_cascade_path)
    print("Face cascade classifier loaded.")
else:
    face_cascade = None
    print(f"WARNING: Face cascade classifier not found at {face_cascade_path}. Face detection will not be available.")

print("Visual models loaded.")


# --- Sub-component: Importance Scoring ---
def calculate_importance(frame_path, ocr_text):
    frame = cv2.imread(frame_path)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h, w = frame.shape[:2]

    # 1. Visual Complexity (edge density)
    edges = cv2.Canny(gray_frame, 100, 200)
    complexity_score = np.sum(edges) / (h * w * 255) # Normalize

    # 2. Text Quantity
    text_score = min(len(ocr_text) / 500.0, 1.0) # Normalize, cap at 500 chars

    # 3. Facial Presence (lower score if face is prominent)
    face_score = 1.0 # Default to 1.0 if face detection is not available
    if face_cascade:
        faces = face_cascade.detectMultiScale(gray_frame, 1.1, 4)
        face_area = sum([fw*fh for (fx,fy,fw,fh) in faces])
        face_score = 1.0 - (face_area / (h * w)) # Inverse score


    # Combine with weights (heuristic, can be tuned)
    final_score = 0.4*complexity_score + 0.4*text_score + 0.2*face_score
    return round(final_score, 2)

# --- Sub-component: Specialized Content Extraction ---

def extract_visual_content(frame_path, frame_type):
    """
    Extracts content from a frame using the best model for the job.
    - Formulas: Uses pix2tex for LaTeX OCR.
    - Code & Text: Uses EasyOCR for robust text recognition.
    - Diagrams: Uses a separate captioning model.
    """
    if frame_type in ['code', 'text']:
        if reader:
            try:
                img_to_ocr = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
                results = reader.readtext(img_to_ocr)
                return '\\n'.join([res[1] for res in results])
            except Exception as e:
                return f"[EasyOCR Error: {e}]"
        else:
             return "Text/Code extraction model not loaded (EasyOCR failed)."

    elif frame_type == 'formula':
        try:
            img = Image.open(frame_path)
            return latex_ocr_model(img)
        except Exception as e:
            return f"[LaTeX-OCR Error: {e}]"

    elif frame_type == 'diagram':
      try:
          img = Image.open(frame_path).convert('RGB')
          return captioning_pipeline(img)[0]['generated_text']
      except Exception as e:
          return f"[Captioning Error: {e}]"

    else:
        return "Content type not supported for extraction."

# --- Worker function for parallel processing ---
def process_single_frame(frame_path, frame_type, timestamp):
    """Function to process one frame; designed to be run in a parallel process."""
    content = extract_visual_content(frame_path, frame_type)
    ocr_text_for_scoring = content if frame_type in ['text', 'code'] else ''
    importance = calculate_importance(frame_path, ocr_text_for_scoring)

    return {
        'timestamp': timestamp,
        'type': frame_type,
        'score': importance,
        'content': content.strip()
    }

# --- Main Visual Processing Function (Optimized) ---
def process_visual_stream(frame_paths):
    """Processes a list of frames to extract all visual information using batching and parallelism."""
    visual_events = []
    if not frame_paths:
        return visual_events

    print(f"Processing {len(frame_paths)} visual frames...")

    # OPTIMIZATION 1: Batch predict all frame types at once.
    if frame_classifier:
        try:
            image_batch = []
            for frame_data in frame_paths:
                img = tf.keras.utils.load_img(frame_data['path'], target_size=IMG_SIZE)
                img_array = tf.keras.utils.img_to_array(img)
                image_batch.append(img_array)

            image_batch = np.array(image_batch)
            predictions = frame_classifier.predict(image_batch, batch_size=BATCH_SIZE, verbose=0)
            predicted_indices = np.argmax(predictions, axis=1)
            frame_types = [CLASS_NAMES[i] for i in predicted_indices]
        except Exception as e:
            print(f"Error during batch classification: {e}. Falling back to unknown type.")
            frame_types = ['unknown'] * len(frame_paths)
    else:
        # Fallback if classifier failed to load
        frame_types = ['unknown'] * len(frame_paths)

    # OPTIMIZATION 2: Process frame content extraction in parallel.
    tasks = []
    for i, frame_data in enumerate(frame_paths):
        tasks.append((frame_data['path'], frame_types[i], frame_data['timestamp']))

    with ProcessPoolExecutor() as executor:
        # Submit all tasks to the process pool
        future_to_frame = {executor.submit(process_single_frame, *task): task for task in tasks}

        for future in as_completed(future_to_frame):
            try:
                result = future.result()
                visual_events.append(result)
            except Exception as exc:
                print(f'Frame processing generated an exception: {exc}')

    # Sort events by timestamp as parallel execution may disorder them
    visual_events.sort(key=lambda x: x['timestamp'])

    print("Visual processing complete.")
    return visual_events

Loading EasyOCR model...
EasyOCR model loaded successfully.


KeyError: "Unknown task image-to-text, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

## Phase 3: Synthesis and Summarization

In [ ]:
# --- Component 3: MAT Assembler ---

def synthesize_mat(spoken_transcript, visual_events):
    """Merges the spoken and visual data streams into a single MAT string."""
    mat_string = ""
    visual_event_idx = 0

    for segment in spoken_transcript:
        segment_start_ms = segment['start'] * 1000
        segment_end_ms = segment['end'] * 1000

        # Add any visual events that occurred before this audio segment
        while visual_event_idx < len(visual_events) and visual_events[visual_event_idx]['timestamp'] < segment_start_ms:
            event = visual_events[visual_event_idx]
            mat_string += f"\n[VISUAL_EVENT @ {int(event['timestamp']/1000)}s]\n"
            mat_string += f"[FRAME_TYPE: {event['type']}]\n"
            mat_string += f"[IMPORTANCE_SCORE: {event['score']}]\n"
            mat_string += f"[CONTENT: {event['content']}]\n\n"
            visual_event_idx += 1

        # Add the spoken text
        mat_string += segment['text'].strip() + " "

    return mat_string

In [ ]:
# --- Component 4: LLM Summarizer ---

# Ensure the gemini_model is initialized before calling these functions
# For example, in a previous cell:
import google.generativeai as genai
from google.colab import userdata
# Use the .strip() method to remove leading/trailing whitespace and newline characters
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY').strip()
genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel('gemini-2.5-pro')


def generate_final_summary(mat_text):
    """Uses an LLM to generate a summary from the rich MAT."""
    print("Querying LLM for final summary...")
    prompt = f"""
    You are an expert academic summarizer. The following is an augmented transcript of a lecture. It includes spoken words and visual events from slides.
    Each [VISUAL_EVENT] has:
    - [FRAME_TYPE]: The type of content ('text', 'diagram', 'formula', 'code').
    - [IMPORTANCE_SCORE]: A score from 0.0 to 1.0 indicating how important the visual is.
    - [CONTENT]: The information extracted from the visual. For formulas, it's in LaTeX format.

    Your task is to create a concise, well-structured summary. Pay close attention to visual events with a high IMPORTANCE_SCORE and accurately incorporate the key formulas, code snippets, and diagram descriptions into your summary.

    --- AUGMENTED TRANSCRIPT ---
    {mat_text}
    --- END TRANSCRIPT ---

    Provide your final, comprehensive summary:
    """

    try:
        # Use the pre-initialized gemini_model
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"LLM Error: Could not generate summary using Google's model. {e}"

# For comparison, a baseline summarizer that only uses the text
def generate_baseline_summary(spoken_transcript):
    print("Generating baseline summary (text only).")
    plain_text = " ".join([seg['text'].strip() for seg in spoken_transcript])
    prompt = f"Summarize the following lecture transcript:\n\n{plain_text}"
    try:
        # Use the pre-initialized gemini_model
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"LLM Error: Could not generate baseline summary using Google's model. {e}"

def generate_quiz(mat_text):
    """Uses an LLM to generate a multiple-choice quiz from the MAT."""
    print("Querying LLM for a multiple-choice quiz...")
    prompt = f"""
    You are an expert quiz creator for educational content. Based on the following augmented transcript of a lecture, generate a multiple-choice quiz that covers the key concepts presented.

    --- AUGMENTED TRANSCRIPT ---
    {mat_text}
    --- END TRANSCRIPT ---

    Please adhere to the following strict formatting and content rules:
    1.  The quiz must have at least 5 multiple-choice questions.
    2.  The questions should be diverse, cover the entire range of topics in the transcript, and not be repetitive.
    3.  The questions must be directly answerable from the provided transcript.
    4.  Do not ask questions about the speaker or presenter. Focus only on the subject matter.
    5.  Ensure all options for a question are unique.
    6.  the question should not talk about transcript.

    Format each question exactly as follows:
    1. Question?
       a) Option 1
       b) Option 2
       c) Option 3
       d) Option 4
    Answer: a

    Note: For the answer, provide ONLY the letter (a, b, c, or d) without any additional text. Use a newline after the answer.

    Begin the quiz now:
    """
    try:
        # Use the pre-initialized gemini_model
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"LLM Error: Could not generate quiz. {e}"

In [ ]:
def generate_reference_summary(mat_text):
    """
    Generates a high-quality, comprehensive reference summary to serve as the
    ground truth for ROUGE score calculation.

    Args:
        mat_text (str): The rich Multimodal-Augmented Transcript.

    Returns:
        str: A detailed, human-quality summary.
    """
    print("Generating Human-Quality Reference Summary for ROUGE evaluation...")

    prompt = f"""
    You are a professional technical writer and subject matter expert. Your task is to create a comprehensive, well-structured, and factually perfect 'gold-standard' summary from the augmented transcript provided below. This summary will be used as the ground truth for evaluating other AI-generated summaries.

    **Instructions:**
    1.  **Be Comprehensive:** Incorporate all critical information from both the spoken text and the [VISUAL_EVENT] annotations. Do not leave out any key concepts, definitions, or examples.
    2.  **Be Structured:** Organize the content logically with clear headings, subheadings, and bullet points to create a perfect study guide.
    3.  **Be Neutral and Formal:** Write in a clear, objective, and academic tone. Do not include any conversational filler.
    4.  **Synthesize, Don't Just List:** Weave the spoken and visual information together seamlessly. For example, if the speaker mentions a command and the screen shows its syntax, present them together.

    **Augmented Transcript:**
    ---
    {mat_text}
    ---

    Produce the 'gold-standard' reference summary now:
    """

    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error generating reference summary: {e}")
        return None # Return None on failure

## Phase 4: Main Execution and Demonstration

Now we tie everything together. Provide a YouTube URL below and run the final cells to see the result.

In [ ]:
# --- Part 1: Data Processing up to MAT Synthesis ---

def process_video_to_mat(video_source):
    """
    Handles video downloading, decomposition, and processing of audio/visual
    streams to generate a Multimodal-Augmented Transcript (MAT).

    Args:
        video_source (str): The YouTube URL or local path of the video.

    Returns:
        tuple: A tuple containing the mat_text (str) and spoken_transcript (list),
               or (None, None) if processing fails.
    """
    # Determine if the source is a URL or a local file
    if video_source.startswith('http'):
        print(f"Downloading video from {video_source}")
        video_path = download_video(video_source, VIDEO_DIR)
    else:
        video_path = video_source

    # Ensure the video file exists before proceeding
    if not video_path or not os.path.exists(video_path):
        print("Video processing failed. File not found.")
        return None, None

    # Step 1: Decompose the video into audio and frame paths
    print("Decomposing video...")
    audio_path, frame_paths = decompose_video(video_path, OUTPUT_DIR)

    # Step 2: Process the audio stream to get the transcript
    print("Processing audio stream...")
    spoken_transcript = process_audio_stream(audio_path)

    # Step 3: Process the visual stream to extract visual events
    print("Processing visual stream...")
    visual_events = process_visual_stream(frame_paths)

    # Step 4: Synthesize the Multimodal-Augmented Transcript (MAT)
    print("Synthesizing MAT...")
    mat_text = synthesize_mat(spoken_transcript, visual_events)

    return mat_text, spoken_transcript

In [ ]:
# --- Part 2: LLM Tasks (Summaries, Quiz) and Display ---

def run_llm_tasks(mat_text, spoken_transcript):
    """
    Uses the generated MAT and spoken transcript to perform all LLM-based tasks,
    including summarization and quiz generation, and displays the results.

    Args:
        mat_text (str): The Multimodal-Augmented Transcript.
        spoken_transcript (list): The list of transcribed audio segments.
    """
    # Fix: Return a tuple of (None, None) on failure for consistent unpacking
    if not mat_text or not spoken_transcript:
        print("Cannot run LLM tasks. Input data is missing.")
        return None, None

    # Step 5: Generate advanced and baseline summaries
    print("Generating summaries...")
    advanced_summary = generate_final_summary(mat_text)
    baseline_summary = generate_baseline_summary(spoken_transcript)

    # Step 6: Generate the multimodal quiz
    print("Generating quiz...")
    quiz = generate_quiz(mat_text)

    # Step 7: Display all the results
    print("\n" + "="*50)
    print("RESULTS")
    print("="*50 + "\n")

    print("--- BASELINE SUMMARY (Text Only) ---")
    print(baseline_summary)
    print("\n" + "-"*50 + "\n")

    print("--- ADVANCED SUMMARY (Multimodal) ---")
    print(advanced_summary)
    print("\n" + "-"*50 + "\n")

    print("--- MULTIMODAL QUIZ ---")
    print(quiz)
    print("\n" + "-"*50 + "\n")

    # Optionally, print a snippet of the MAT to see the structure
    print("--- MAT Snippet ---")
    # To keep the output clean, show only the first 1000 characters
    print(mat_text[:1000] + "...")

    # This return statement is now safe
    return advanced_summary, quiz

## Phase 5: Evaluation

This phase contains all functions related to evaluating the generated content. It includes:
1.  **Quantitative Metrics**: Using ROUGE scores to provide a traditional, objective measure of summary quality against a reference text.
2.  **Qualitative Metrics**: Using the advanced LLM-as-a-Judge method for a nuanced, critical assessment of both the summary and the quiz.

In [ ]:
# --- Evaluation Functions ---

from rouge_score import rouge_scorer
import google.generativeai as genai
import json
import re

# --- Placeholder for Ground Truth Data ---
# For a real research paper, this REFERENCE_SUMMARY would be a high-quality,
# human-written summary of the specific video being tested. It serves as the
# ground truth for calculating ROUGE scores.
REFERENCE_SUMMARY = """
This lecture introduces Structured Query Language (SQL) as the primary language for interacting with relational databases via a Database Management System (DBMS), such as MySQL or Oracle. The core of database manipulation is defined by CRUD operations: Create (INSERT), Read (SELECT), Update (UPDATE), and Delete (DELETE). The tutorial explains the syntax for these commands, highlighting the use of the WHERE clause for filtering and the JOIN clause for combining data from multiple tables.

To ensure data integrity, the system uses constraints like PRIMARY KEY and FOREIGN KEY. For performance, it utilizes Indexes. Data analysis is facilitated by aggregate functions (e.g., SUM, COUNT) which are used in conjunction with the GROUP BY clause to summarize data. Results can be sorted using the ORDER BY clause.

Finally, the lecture briefly touches on advanced concepts, including Views (virtual tables to simplify queries), Transactions (for ensuring data consistency via ACID properties), Stored Procedures (pre-compiled server-side code), and Triggers (automated procedures that run in response to specific database events).
"""


def evaluate_rouge(generated_summary, reference_summary):
    """
    Calculates ROUGE-1, ROUGE-2, and ROUGE-L F1-scores for a generated summary.

    Args:
        generated_summary (str): The summary produced by the model.
        reference_summary (str): The ground truth summary.

    Returns:
        dict: A dictionary containing the ROUGE scores.
    """
    if not reference_summary:
        print("Reference summary not provided. Skipping ROUGE evaluation.")
        return {'rouge-1': 'N/A', 'rouge-2': 'N/A', 'rouge-l': 'N/A'}

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference_summary, generated_summary)

    return {
        'rouge-1': round(scores['rouge1'].fmeasure, 3),
        'rouge-2': round(scores['rouge2'].fmeasure, 3),
        'rouge-l': round(scores['rougeL'].fmeasure, 3)
    }

def call_llm_judge(prompt):
    """
    Helper function to call the LLM judge and parse its JSON response.
    """
    try:
        response = gemini_model.generate_content(prompt)
        return json.loads(response.text)
    except Exception as e:
        print(f"Error calling LLM Judge or parsing response: {e}")
        return {"error": f"LLM Judge call failed: {e}"}

# (Paste your entire evaluate_with_llm_judge function here as well)
def evaluate_with_llm_judge(advanced_summary, quiz, mat_text):
    """
    Uses a powerful LLM to perform a STRICT, high-resolution evaluation using decimal scores.
    """
    print("Querying STRICT LLM Judge for critical, high-resolution evaluation...")

    # --- ENHANCED Prompt for Summary Evaluation with Decimal Scoring ---
    summary_prompt = f"""
    You are a skeptical and meticulous academic reviewer. Your task is to provide a critical assessment of an AI-generated summary against a source transcript using a high-resolution decimal scale. Be discerning and avoid awarding high scores unless the output is truly exceptional.

    **Scoring Rubric (1.0 to 5.0 scale):**
    - **5.0 (Exceptional):** Flawless, professional, publication-ready quality. Cannot be improved.
    - **4.0-4.9 (Excellent):** Very high quality with only minor stylistic or subtle rooms for improvement.
    - **3.0-3.9 (Good):** Competent and correct. It successfully completes the task but has clear areas for improvement in structure, detail, or flow. This is the baseline for a "successful" generation.
    - **2.0-2.9 (Acceptable):** Contains noticeable errors, omissions, or structural problems, but still captures the main gist.
    - **1.0-1.9 (Poor):** Fails to meet the basic requirements of the task.

    **Evaluation Criteria & Instructions:**
    For each criterion, provide a score as a float (e.g., 3.5, 4.8), a justification, and a concrete 'suggested_improvement'. You must suggest an improvement unless the score is a perfect 5.0.

    1.  **Relevance:** How well does it capture the most critical points from both spoken and visual content?
    2.  **Conciseness:** How well does it avoid redundant information and filler?
    3.  **Coherence:** How logical and easy to follow is the structure?
    4.  **Factual Accuracy:** Does it perfectly represent all facts and technical details without error?

    **Input Transcript:**
    ---
    {mat_text}
    ---
    **Generated Summary to Evaluate:**
    ---
    {advanced_summary}
    ---

    Provide your evaluation in a strict JSON format with float values for scores.

    **Required JSON Format:**
    ```json
    {{
      "summary_evaluation": {{
        "relevance": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }},
        "conciseness": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }},
        "coherence": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }},
        "factual_accuracy": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }}
      }}
    }}
    ```
    """

    # --- ENHANCED Prompt for Quiz Evaluation with Decimal Scoring ---
    quiz_prompt = f"""
    You are a skeptical and meticulous academic reviewer specializing in educational assessment. Critique a multiple-choice quiz using a high-resolution decimal scale. Hold the quiz to a high pedagogical standard.

    **Scoring Rubric (1.0 to 5.0 scale):**
    - **5.0 (Exceptional):** Flawless pedagogical quality. Insightful questions and perfectly crafted distractors.
    - **4.0-4.9 (Excellent):** High quality with minor room for improvement in question phrasing or distractor nuance.
    - **3.0-3.9 (Good):** The quiz is functional and effective but could be more challenging or better targeted.
    - **2.0-2.9 (Acceptable):** Contains unclear questions, obvious answers, or minor errors.
    - **1.0-1.9 (Poor):** Not a useful assessment tool.

    **Evaluation Criteria & Instructions:**
    For each criterion, provide a score as a float (e.g., 3.5, 4.8), a justification, and a concrete 'suggested_improvement'. You must suggest an improvement unless the score is a perfect 5.0.

    1.  **Clarity & Unambiguity:** Are the questions and options perfectly clear?
    2.  **Relevance to Content:** Do the questions target the most important concepts and avoid trivia?
    3.  **Correctness of Answers:** Are the answers unequivocally correct based on the transcript?
    4.  **Plausibility of Distractors:** Are the incorrect options challenging and believable?

    **Input Transcript:**
    ---
    {mat_text}
    ---
    **Generated Quiz to Evaluate:**
    ---
    {quiz}
    ---

    Provide your evaluation in a strict JSON format with float values for scores.

    **Required JSON Format:**
    ```json
    {{
      "quiz_evaluation": {{
        "clarity": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }},
        "relevance": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }},
        "correctness": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }},
        "plausibility_of_distractors": {{
          "score": <float>,
          "justification": "<Your reasoning>",
          "suggested_improvement": "<A specific suggestion>"
        }}
      }}
    }}
    ```
    """
    # (The rest of your existing evaluate_with_llm_judge function...)
    summary_evaluation = call_llm_judge(summary_prompt)
    quiz_evaluation = call_llm_judge(quiz_prompt)

    final_evaluation = {
        "summary_evaluation": summary_evaluation.get("summary_evaluation", {"error": "Evaluation could not be parsed."}),
        "quiz_evaluation": quiz_evaluation.get("quiz_evaluation", {"error": "Evaluation could not be parsed."})
    }

    print("LLM Judge critical, high-resolution evaluation complete.")
    return final_evaluation


In [ ]:
# --- Main Execution Block (Updated for Dynamic Reference Summary) ---

def run_full_pipeline_with_evaluation(video_source):
    """
    Executes the entire pipeline: processing, generation, and a two-pronged evaluation
    (quantitative ROUGE and qualitative LLM-as-a-Judge).
    """
    # --- Part 1: Data Processing ---
    mat_text, spoken_transcript = process_video_to_mat(video_source)

    if not mat_text or not spoken_transcript:
        print("Pipeline execution halted due to an error in video processing.")
        return

    # --- Part 2: Generate Ground Truth and Model Outputs ---

    # NEW STEP: Dynamically generate the reference summary for this video
    REFERENCE_SUMMARY = generate_reference_summary(mat_text)
    if not REFERENCE_SUMMARY:
        print("Could not generate a reference summary. Halting evaluation.")
        return

    # Generate the summaries and quiz to be evaluated
    print("Generating summaries and quiz for evaluation...")
    advanced_summary = generate_final_summary(mat_text)
    baseline_summary = generate_baseline_summary(spoken_transcript)
    quiz = generate_quiz(mat_text)

    if not advanced_summary or not quiz:
        print("Content generation failed. Halting evaluation.")
        return

    # --- Part 3: Evaluation ---
    print("\n--- Running Evaluations ---")
    # 3a. Quantitative Evaluation (ROUGE) using the dynamic reference
    rouge_advanced = evaluate_rouge(advanced_summary, REFERENCE_SUMMARY)
    rouge_baseline = evaluate_rouge(baseline_summary, REFERENCE_SUMMARY)

    # 3b. Qualitative Evaluation (LLM-as-a-Judge)
    llm_evaluation = evaluate_with_llm_judge(advanced_summary, quiz, mat_text)

    # --- Part 4: Display Final Report ---
    print("\n\n" + "="*60)
    print(" " * 20 + "FINAL EVALUATION REPORT")
    print("="*60)

    # Generated Content
    print("\n--- ADVANCED SUMMARY (Multimodal) ---")
    print(advanced_summary)
    print("\n--- BASELINE SUMMARY (Audio-Only) ---")
    print(baseline_summary)
    print("\n--- MULTIMODAL QUIZ ---")
    print(quiz)

    # Quantitative Metrics First
    print("\n" + "="*60)
    print(" " * 18 + "1. Quantitative Metrics (ROUGE)")
    print("="*60)
    print("\nBaseline Summary (Audio-Only):")
    print(f"  - ROUGE-1: {rouge_baseline['rouge-1']}")
    print(f"  - ROUGE-2: {rouge_baseline['rouge-2']}")
    print(f"  - ROUGE-L: {rouge_baseline['rouge-l']}")
    print("\nAdvanced Summary (Multimodal):")
    print(f"  - ROUGE-1: {rouge_advanced['rouge-1']}")
    print(f"  - ROUGE-2: {rouge_advanced['rouge-2']}")
    print(f"  - ROUGE-L: {rouge_advanced['rouge-l']}")

    # Qualitative Metrics Second
    print("\n" + "="*60)
    print(" " * 17 + "2. Qualitative Metrics (LLM-as-a-Judge)")
    print("="*60)
    print("\n--- Summary Evaluation (Multimodal) ---")
    print(json.dumps(llm_evaluation.get('summary_evaluation'), indent=2))
    print("\n--- Quiz Evaluation ---")
    print(json.dumps(llm_evaluation.get('quiz_evaluation'), indent=2))
    print("\n" + "="*60)


# --- Run Everything ---
YOUTUBE_URL = "https://www.youtube.com/watch?v=pBASqUbZgkY" # SQL Tutorial video
video_source = YOUTUBE_URL

run_full_pipeline_with_evaluation(video_source)